In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from subprocess import run, PIPE
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
import json

### Check len

In [ ]:
DATA_PATH = Path("/mnt/data/kusnierz/audio-data/SingFake/downloads")
log_file = Path("/mnt/data/kusnierz/audio-data/SingFake/download_log.json")
original_csv = Path("/mnt/data/kusnierz/audio-data/SingFake/singfake.csv")

### base csv data splits

In [ ]:
# Read the CSV file
df_csv = pd.read_csv(original_csv, header=None, names=["split", "label", "lang", "artist", "title", "unknown", "url"])

# Count spoof vs bonafide for each split
stats = df_csv.groupby(["split", "label"]).size().reset_index(name="count")
print(stats)

In [ ]:
df_csv.groupby(['split'])['lang'].value_counts()

In [ ]:
df_csv['lang'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# General language distribution
lang_counts = df_csv['lang'].value_counts()
plt.figure(figsize=(10,7))
ax = sns.barplot(x=lang_counts.index, y=lang_counts.values)
plt.xticks(rotation=45)
plt.title('Language Distribution (General)')
plt.ylabel('Count')
plt.xlabel('Language')
plt.tight_layout()
# Add exact numbers on top of bars
for i, v in enumerate(lang_counts.values):
    ax.text(i, v + max(lang_counts.values)*0.01, str(v), ha='center', va='bottom', fontweight='bold')
plt.show()

# Split-wise language distribution
split_lang_counts = df_csv.groupby(['split'])['lang'].value_counts().unstack(fill_value=0)
split_lang_counts.plot(kind='bar', stacked=True, figsize=(12,6))
plt.title('Language Distribution by Split')
plt.ylabel('Count')
plt.xlabel('Split')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### downloading success

In [ ]:
with open(log_file, "r") as file:
    json_data = json.load(file)

success = 0
errors = 0  
for item in json_data:
    if item['status'] == "success":
        success+=1
    else:
        errors+=1
        
print(f"successful: {success}")
print(f"failed: {errors}")

In [ ]:
files = [p for p in DATA_PATH.rglob("*") if p.is_file() and p.suffix.lower() == ".flac"]

def ffprobe_duration(path: Path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ]
    proc = run(cmd, stdout=PIPE, stderr=PIPE, text=True)
    if proc.returncode != 0 or not proc.stdout.strip():
        return {"error": True, "filepath": str(path), "error_msg": proc.stderr.strip() or "no-duration"}
    try:
        dur = float(proc.stdout.strip())
        return {"filename": path.stem, "filepath": str(path), "duration": int(dur)}
    except Exception as e:
        return {"error": True, "filepath": str(path), "error_msg": repr(e)}

results = []
failed = []
max_workers = min(32, max(4, (os.cpu_count() or 4) * 2))
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(ffprobe_duration, p): p for p in files}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        res = fut.result()
        if res.get("error"):
            failed.append((res["filepath"], res["error_msg"]))
        else:
            results.append(res)

df = pd.DataFrame(results)
failed_df = pd.DataFrame(failed, columns=["filepath", "error"])
print("success:", len(df), "failed:", len(failed_df))

In [ ]:
def extract_info(row):
    parts = row['filename'].split('_')
    split = parts[0] if len(parts) > 1 else 'Unknown'
    label = 'bonafide' if 'bonafide' in row['filename'].lower() else (
        'spoof' if 'spoof' in row['filename'].lower() else 'Unknown'
    )
    return pd.Series({'split': split, 'label': label})

df[['split', 'label']] = df.apply(extract_info, axis=1)
count_df = df.groupby(['split', 'label']).size().reset_index(name='count')
print(count_df)

In [ ]:
df.groupby(['label']).count()

In [ ]:
df.describe()

In [ ]:
filtered_df = df[df['duration'] < 1000]

In [ ]:
filtered_df

In [ ]:
filtered_df.describe()

In [ ]:
sns.histplot(filtered_df, x='duration', bins=100)

In [ ]:
tmp_path = df.iloc[10]['filepath']

In [ ]:
import torchaudio

In [ ]:
wav, sr = torchaudio.load(tmp_path)

resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
wav = resampler(wav)

spectrogram_layer = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_fft=2048, n_mels=128, hop_length=160)
spectrogram = spectrogram_layer(wav)

In [ ]:
spectrogram.shape

In [ ]:
part = spectrogram[0, :, :4096]

In [ ]:
S_db = torchaudio.transforms.AmplitudeToDB()(part)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(15,8))
sns.heatmap(S_db)

### Check data after resample + spectrograms + normalization

In [ ]:
import rootutils
ROOT = rootutils.setup_root(".", indicator=".project-root", pythonpath=True)

In [ ]:
from src.data.singfake_dataloader import SingFakeDataModule

In [ ]:
spectrogram_path = Path("/mnt/data/kusnierz/audio-data/SingFake/spectrograms")
specs_list = list(spectrogram_path.rglob("*.pt"))

In [ ]:
spec = torch.load(specs_list[20])

In [ ]:
spec.shape

In [ ]:
params = {"sample_rate": 16000,
"n_fft": 1024,
"n_mels": 80,
"hop_length": 512,
"max_len": 8000000}

In [ ]:
dataloader = SingFakeDataModule(
    "/mnt/data/kusnierz/audio-data/SingFake/spectrograms",
    16,
    8,
    **params
    )

In [ ]:
dataloader.setup(stage='fit')

In [ ]:
dataset = dataloader.training_dataset

In [ ]:
obj, label = dataset.__getitem__(20)

In [ ]:
obj.shape

In [ ]:
obj.size(0)

In [ ]:
plt.figure(figsize=(15,8))
sns.heatmap(obj.transpose(0,1))

In [ ]:
import random

LABEL_MAP = {"bonafide": 1, "spoof": 0}

def infer_label_from_stem(stem: str):
    parts = stem.split("_")
    if len(parts) == 0:
        return None
    last = parts[-1].lower()
    return last if last in LABEL_MAP else None

def pick_files(data_dir: Path, mode: str, n_bona: int, n_spoof: int, seed: int = 42):
    random.seed(seed)
    files = [p for p in data_dir.rglob("*") if p.is_file()]
    if mode == "spec":
        files = [p for p in files if p.suffix.lower() == ".pt"]
    else:
        files = [p for p in files if p.suffix.lower() in {".flac", ".wav", ".mp3", ".m4a", ".ogg"}]

    bona = []
    spoof = []
    for p in files:
        lab = infer_label_from_stem(p.stem)
        if lab == "bonafide":
            bona.append(p)
        elif lab == "spoof":
            spoof.append(p)

    if len(bona) < n_bona or len(spoof) < n_spoof:
        raise ValueError(f"Not enough samples found (bona: {len(bona)}, spoof: {len(spoof)})")

    return random.sample(bona, n_bona), random.sample(spoof, n_spoof)

def load_spec_for_plot(path: Path):
    t = torch.load(path, map_location="cpu", weights_only=False)
    if not torch.is_tensor(t):
        raise RuntimeError(f"{path} did not contain a tensor")
    t = t.squeeze()
    if t.ndim != 2:
        raise RuntimeError(f"Unsupported tensor shape {t.shape} in {path}")
    # If saved as [Time, Feat] -> transpose to [Freq, Time] for plotting.
    if t.shape[0] > t.shape[1]:
        arr = t.T.numpy()
    else:
        arr = t.numpy()
    return arr  # shape: [freq, time] for imshow

def load_audio(path: Path, target_sr: int = 16000):
    wav, sr = torchaudio.load(path)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        wav = resampler(wav)
    return wav, target_sr

def concat_audio(paths, out_path: Path, target_sr: int = 16000):
    parts = []
    for p in paths:
        wav, sr = load_audio(p, target_sr=target_sr)
        parts.append(wav)
    combined = torch.cat(parts, dim=1)
    torchaudio.save(str(out_path), combined, target_sr)
    return out_path

In [ ]:
DATA_DIR = Path("/mnt/data/kusnierz/audio-data/SingFake/spectrograms")  # change if needed
MODE = "spec"   # "spec" or "audio"
N_BONA = 5
N_SPOOF = 5
SEED = 123

bona_files, spoof_files = pick_files(DATA_DIR, MODE, N_BONA, N_SPOOF, seed=SEED)
print("Bona samples:")
for p in bona_files: print(" ", p.name)
print("Spoof samples:")
for p in spoof_files: print(" ", p.name)

In [ ]:
def plot_bona_spoof(bona_paths, spoof_paths, figsize_per_col=(4,3), cmap="magma", vmin=None, vmax=None):
    cols = max(len(bona_paths), len(spoof_paths))
    fig, axes = plt.subplots(2, cols, figsize=(figsize_per_col[0]*cols, figsize_per_col[1]*2), constrained_layout=True)
    if cols == 1:
        axes = np.expand_dims(axes, axis=1)
    for c in range(cols):
        # Bona
        ax_b = axes[0, c]
        if c < len(bona_paths):
            arr = load_spec_for_plot(bona_paths[c])
            im = ax_b.imshow(arr, aspect="auto", origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
            ax_b.set_title("Bona", fontsize=8)
        else:
            ax_b.axis("off")
        # Spoof
        ax_s = axes[1, c]
        if c < len(spoof_paths):
            arr = load_spec_for_plot(spoof_paths[c])
            ax_s.imshow(arr, aspect="auto", origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
            ax_s.set_title("Spoof", fontsize=8)
        else:
            ax_s.axis("off")
        ax_s.set_xlabel("Time")
    # colorbar using last image
    fig.colorbar(im, ax=axes.ravel().tolist(), orientation="vertical", fraction=0.02)
    plt.show()

# Call plotting
plot_bona_spoof(bona_files, spoof_files)

In [ ]:
if MODE == "audio":
    out = concat_audio(bona_files + spoof_files, Path("combined_bona_spoof.wav"))
    print("Saved combined audio to", out)
else:
    print("Skipping audio concat (MODE != 'audio').")

## Split data

In [ ]:
dir_path = Path("/mnt/data/kusnierz/audio-data/SingFake/data_after_separation/split_dump")

In [ ]:

subfolders = ["mixtures", "vocals"]
results = []

for sub in subfolders:
    folder = dir_path / sub
    files = [f for f in folder.rglob("*.flac") if f.is_file()]
    for f in files:
        name = f.stem
        # Split name: first part is split, last part before __seg is label
        parts = name.split("_")
        split = parts[0] if len(parts) > 1 else "Unknown"
        label = "bonafide" if "bonafide" in name.lower() else ("spoof" if "spoof" in name.lower() else "Unknown")
        results.append({"subfolder": sub, "split": split, "label": label})

# Convert to DataFrame for easy counting
results_df = pd.DataFrame(results)

# Count files per split and label for each subfolder
for sub in subfolders:
    print(f"--- {sub} ---")
    sub_df = results_df[results_df["subfolder"] == sub]
    print("Total files:", len(sub_df))
    print(sub_df.groupby(["split", "label"]).size().reset_index(name="count"))
